# 5.16 配比怎麼落到實際 batch？


你想讓模型九成時間練短答、一成練長答，按筆數抽90筆短答與10筆長答。可是短答每筆只有一個答案token，長答每筆有二十個；算每個有效答案的平均loss時，實際長答貢獻200題、短答90題。筆數比例90:10，不會自動成為監督token比例90:10。

[5.2](https://birdhackor.github.io/tiny-perceptron-vlm/5.2.html) 已確定代價分母按有效答案計數，本節把這個規則用在資料混合。source（來源）是每筆資料的類別標記，例如短答、長答，讓我們能在組batch後統計實際用了哪些資料。光記「預期抽樣九成」還不夠，要檢查執行出來的數量與長度。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
counts = {"短答": 90, "長答": 10}
lengths = {"短答": 1, "長答": 20}
tokens = {}
for source in counts:
    tokens[source] = counts[source] * lengths[source]
total = sum(tokens.values())
ratios = {source: amount / total for source, amount in tokens.items()}
print("有效答案數", tokens)
print("有效答案比例", ratios)

輸入counts是筆數、lengths是假設每筆有效答案數。迴圈逐來源相乘，得到短答90、長答200，總共290。輸出比例約0.310與0.690，所以少數長答貢獻約七成監督。這裡只計答案位置，不把用戶問題、填充或被忽略的文字混進去；真正batch長度不齊時，要逐筆數有效label，而不是用補齊後的總格數。

如果目標真的是token比例90:10，需要調整抽樣筆數、每筆取多少有效內容，或明確改變來源loss權重；這些會改變訓練目標，不能僅把記錄裡的百分比改漂亮。假設兩類固定長度1與20，要令長答token只佔一成，短答筆數約應為長答筆數的180倍。你能從 `短數×1 : 長數×20 = 9:1` 算出這個比例，但數據與batch實際可行性還需檢查。

來源比例也可能隨訓練階段變化，長答平均長度也會漂移，所以要記錄實際累計token而非一次配置。評估時分來源報指標，可以看出總平均改善是不是只因為最常見來源佔得更多，少數任務反而退步。資料預算的約定見 [5.13](https://birdhackor.github.io/tiny-perceptron-vlm/5.13.html)。

這個例子用假設每筆相同長度讓數值可手算，真資料可能長短分佈很寬。若每批剛好只有一兩筆長答，短期比例波動大，應統計一段訓練累計，並說明是否重新抽取或重複抽取某類資料（重採樣），以及是否只保留每筆答案的一部分（截斷）。前者會改實際筆數，後者會改有效答案位置數，都會影響最後比例。

這個差別已在[7.16的回放實驗](https://birdhackor.github.io/tiny-perceptron-vlm/7.16.html)實際出現：兩支都更新500次、每批16題，只練加法的累計有效答案是18,453個，混回較長屬性答案的支線卻有36,384個。配方的筆數與步數相同，監督總量仍差近一倍。因此比較資料混合成績時，要把實際計分位置帶回來，不能只寫「都跑500步」。

練習只把長答長度20改成1，先預測兩類token為90與10、比例回到0.9與0.1，再跑核對。筆數完全沒變，變化來自答案長度。再恢復20並把長答筆數改成5，手算長答100、總190、比例約0.526，驗證少一半長答仍可能主導監督。
